In [ ]:
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
BUCKET_NAME = "your-gcs-bucket-name"
PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline_root/"

In [ ]:
# Create the bucket
!gsutil mb -l {REGION} gs://{BUCKET_NAME}

In [ ]:
import json
import pickle
import pandas as pd
import numpy as np
from typing import NamedTuple
from google.cloud import storage
from kfp.v2 import dsl
from kfp.v2.dsl import (
    component, Artifact, Dataset, Input, Model, Output, Metrics
)
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Define components
@component
def ingest_data() -> NamedTuple("Outputs", [("X", list), ("y", list)]):
    iris = load_iris()
    return iris.data.tolist(), iris.target.tolist()

@component
def split_data(X: list, y: list) -> NamedTuple("Outputs", [("X_train", list), ("X_test", list), ("y_train", list), ("y_test", list)]):
    X_train, X_test, y_train, y_test = train_test_split(np.array(X), np.array(y), test_size=0.2)
    return X_train.tolist(), X_test.tolist(), y_train.tolist(), y_test.tolist()

@component
def train_model(X_train: list, y_train: list, max_depth: int = 2, n_estimators: int = 100, random_state: int = 0) -> str:
    clf = RandomForestClassifier(max_depth=max_depth, n_estimators=n_estimators, random_state=random_state)
    clf.fit(np.array(X_train), np.array(y_train))
    model_path = "model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(clf, f)
    return model_path

@component
def evaluate_model(model_path: str, X_test: list, y_test: list, metrics: Output[Metrics]):
    with open(model_path, "rb") as f:
        clf = pickle.load(f)
    y_pred = clf.predict(np.array(X_test))
    report = classification_report(np.array(y_test), y_pred, output_dict=True)
    cm = confusion_matrix(np.array(y_test), y_pred)
    print("Confusion Matrix:", cm)
    print("Classification Report:", report)
    
    metrics.log_metric("accuracy", report["accuracy"])
    metrics.log_metric("precision", report["weighted avg"]["precision"])
    metrics.log_metric("recall", report["weighted avg"]["recall"])
    metrics.log_metric("f1-score", report["weighted avg"]["f1-score"])

@component
def register_model(model_path: str, bucket_name: str) -> str:
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob("models/random_forest/model.pkl")
    blob.upload_from_filename(model_path)
    return f"gs://{bucket_name}/models/random_forest/model.pkl"

# Define pipeline
def train_pipeline(bucket_name: str):
    @dsl.pipeline(
        name="random-forest-training-pipeline",
        description="A pipeline that trains a RandomForest model on the Iris dataset"
    )
    def pipeline():
        data = ingest_data()
        split = split_data(data.outputs["X"], data.outputs["y"])
        model = train_model(split.outputs["X_train"], split.outputs["y_train"])
        evaluate_model(model.outputs["model_path"], split.outputs["X_test"], split.outputs["y_test"])
        register_model(model.outputs["model_path"], bucket_name)
    return pipeline

# Run the pipeline
if __name__ == "__main__":
    from kfp.v2.google.client import AIPlatformClient
    
    client = AIPlatformClient(project_id=PROJECT_ID, region=REGION)
    
    client.create_run_from_pipeline_func(
        train_pipeline(BUCKET_NAME),
        pipeline_root=PIPELINE_ROOT,
        parameter_values={}
    )
